# 05 - Article figures

Research question: can a single notebook generate a compact article-ready figure bundle from the repository tools?

Data dependencies: none. This notebook uses only synthetic simulations.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dynsys_econometrics.extremes import estimate_runs_extremal_index, extract_threshold_exceedances
from dynsys_econometrics.return_times import compute_return_times
from dynsys_econometrics.simulation import logistic_map, simulate_ar1, simulate_garch11, simulate_iid_gaussian, simulate_student_t


In [ ]:
n_steps = 3500
threshold_quantile = 0.97

series = {
    "iid": simulate_iid_gaussian(n_steps, seed=21),
    "ar1": simulate_ar1(n_steps, phi=0.5, sigma=1.0, seed=21),
    "garch11": simulate_garch11(n_steps, seed=21),
    "logistic": logistic_map(n_steps, x0=0.33, r=3.99),
    "student_t": simulate_student_t(n_steps, df=6, seed=21),
}

rows = []
for name, values in series.items():
    threshold = float(np.quantile(values, threshold_quantile))
    exceedances = extract_threshold_exceedances(values, threshold)
    result = estimate_runs_extremal_index(values, threshold_quantile=threshold_quantile, run_length=4)
    return_times = compute_return_times(values, threshold)
    rows.append(
        {
            "series": name,
            "n_exceedances": exceedances.values.size,
            "theta_runs": result.theta_hat,
            "mean_return_time": float(return_times.mean()) if return_times.size else float("nan"),
        }
    )

summary = pd.DataFrame(rows)
summary


In [ ]:
output_dir = Path("article/figures")
output_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(2, 1, figsize=(11, 8))
summary.plot(x="series", y="theta_runs", kind="bar", legend=False, ax=axes[0], color="#4c78a8")
axes[0].set_title("Extremal index summary")
axes[0].set_xlabel("series")
axes[0].set_ylabel("theta")

summary.plot(x="series", y="mean_return_time", kind="bar", legend=False, ax=axes[1], color="#f58518")
axes[1].set_title("Mean return time summary")
axes[1].set_xlabel("series")
axes[1].set_ylabel("mean return time")

fig.tight_layout()
fig.savefig(output_dir / "05_article_figures_summary.png", dpi=160)
plt.close(fig)

fig2, ax2 = plt.subplots(figsize=(10, 4))
for name, values in series.items():
    threshold = float(np.quantile(values, threshold_quantile))
    exceedances = extract_threshold_exceedances(values, threshold)
    ax2.scatter(exceedances.indices, values[exceedances.indices], s=8, alpha=0.7, label=name)
ax2.set_title("Threshold exceedance locations")
ax2.set_xlabel("time index")
ax2.set_ylabel("value")
ax2.legend(loc="upper right", fontsize="small")
fig2.tight_layout()
fig2.savefig(output_dir / "05_article_figures_exceedances.png", dpi=160)
plt.close(fig2)

[
    str(output_dir / "05_article_figures_summary.png"),
    str(output_dir / "05_article_figures_exceedances.png"),
]
